# SAMap supplementary figure, panel e: transferred against human pseudotime, by compartment

For every human nucleus with both values, the pseudotime transferred from mouse (y) against the human pseudotime
(x): all nuclei, then decidual cells and fibroblasts alone. Spearman's rho is printed in each plot; the all-cells
plot adds the median of both values per decile of human pseudotime. Statistics use all nuclei.

**Input**: `human_cells.csv`, written by rule `samap_figure_tables`: human pseudotime (`pt_human`), the transfer
of record (`pt_transferred`) and the compartment (`cell_type`).
**Outputs**, in `OUT`: `sfig_samap_panel_e.png` (300 dpi), `sfig_samap_panel_e_600dpi.png`,
`sfig_samap_panel_e.svg` (text kept as text) and `sfig_samap_panel_e.provenance.json`.
`sfig_samap_compose.ipynb` places the panel in the figure and adds its letter.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_e_transferred_vs_human.ipynb \
    results/notebooks/samap/sfig_samap_panel_e.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
HUMAN_CELLS = "results/samap_figures/tables/human_cells.csv"
OUT = "results/figures/samap_supp/panels"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_e_transferred_vs_human.ipynb"  # this notebook, for the provenance record

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
from IPython.display import Image, display

import sfig_samap_style as style

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_e")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
LETTER = "e"
TITLE = "Transferred pseudotime preserves human ordering\nwithin decidual cells, weakly within fibroblasts"
WIDTH_SHOWN = 600  # px, the PNG shown in this notebook

# The three scatter plots, left to right: name, compartment (None for all nuclei) and point color (None: by
# compartment).
SUBSETS = [
    ("all human cells", None, None),
    ("decidual cells", "decidual_cell", style.DECIDUAL),
    ("fibroblasts", "endometrial_fibroblast", style.FIBROBLAST),
]

# About 47,000 points per plot: small, translucent dots show density without binning.
POINT_SIZE = 0.25  # pt^2
POINT_ALPHA = 0.25
N_BINS = 10  # deciles of human pseudotime for the median trend

## Statistics

In [ ]:
def spearman_by_subset(subsets: dict) -> dict:
    """Spearman's rho between human and transferred pseudotime in each subset."""
    return {name: spearmanr(cells.pt_human, cells.pt_ranknorm).statistic for name, cells in subsets.items()}

## Figure

In [ ]:
def draw_transferred_vs_human(subsets: dict, rho: dict):
    fig, width_mm, height_mm = style.new_panel(LETTER)

    # Three square-ish plots sharing the y axis: 18 mm left for the label, 3.5 mm between plots.
    x0, gap, total = 18 / width_mm, 3.5 / width_mm, 1 - 22 / width_mm
    axes_w = (total - 2 * gap) / 3
    axes_y, axes_h = 12.0 / height_mm, 1 - 27.0 / height_mm
    axs = [fig.add_axes([x0 + i * (axes_w + gap), axes_y, axes_w, axes_h]) for i in range(3)]

    for i, (ax, (name, _, color)) in enumerate(zip(axs, SUBSETS)):
        cells = subsets[name]

        # All nuclei in a seeded random order, colored by compartment, so neither compartment always sits on top.
        if color is None:
            shown = cells.sample(frac=1, random_state=4)
            point_colors = shown.cell_type.map(style.COMPARTMENT_COLORS).values
        else:
            shown = cells
            point_colors = color
        ax.scatter(shown.pt_human, shown.pt_ranknorm, s=POINT_SIZE, c=point_colors, alpha=POINT_ALPHA,
                   linewidths=0, rasterized=True)

        # Median trend only on the all-nuclei plot; the compartment plots show the points alone.
        if color is None:
            decile = pd.qcut(cells.pt_human.rank(method="first"), N_BINS, labels=False)
            medians = cells.groupby(decile)[["pt_human", "pt_ranknorm"]].median()
            ax.plot(medians.pt_human, medians.pt_ranknorm, color="black", lw=0.9, zorder=4)
            ax.scatter(medians.pt_human, medians.pt_ranknorm, marker="D", s=9, color="black",
                       edgecolor="white", linewidths=0.4, zorder=5)
            ax.legend(
                handles=[Line2D([], [], color="black", lw=0.9, marker="D", ms=2.8, mec="white", mew=0.4,
                                label="median per decile")],
                loc="lower right",
                frameon=False,
                fontsize=7,
                handlelength=1.6,
                borderaxespad=0.2,
                handletextpad=0.4,
            )

        ax.text(0.04, 0.96, f"ρ = {rho[name]:.2f}\nn = {len(cells) / 1e3:.1f}k",
                transform=ax.transAxes, va="top", ha="left", fontsize=8)
        ax.set_title(name, loc="left", fontsize=8, pad=2)

        # Both values are ranks scaled to 0-1; 3% margins keep edge points inside the frame.
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xticks([0, 0.5, 1], labels=["0", "0.5", "1"])
        ax.set_yticks([0, 0.5, 1], labels=["0", "0.5", "1"])
        if i:
            ax.set_yticklabels([])

    axs[0].set_ylabel("transferred from mouse")
    axs[1].set_xlabel("human pseudotime (rank-normalized)")
    style.panel_title(fig, width_mm, height_mm, TITLE)
    return fig

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

paths = style.panel_paths(repo / OUT, LETTER)
style.refuse_overwrite(paths.values(), FORCE)
paths["png"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
human = pd.read_csv(repo / HUMAN_CELLS, index_col="cell")
values = human[["pt_transferred", "pt_human", "cell_type"]].rename(columns={"pt_transferred": "pt_ranknorm"})

# Nuclei with both values; the human nuclei without a valued mouse neighbor drop out here.
pairs = values[values.pt_human.notna() & values.pt_ranknorm.notna()]
LOG.info("Human nuclei with both values: %d of %d", len(pairs), len(values))

subsets = {
    name: pairs if cell_type is None else pairs[pairs.cell_type == cell_type]
    for name, cell_type, _ in SUBSETS
}
rho = spearman_by_subset(subsets)
summary = pd.DataFrame({"n": {name: len(cells) for name, cells in subsets.items()}, "spearman_rho": rho})
display(summary.round(4))

In [ ]:
font = style.choose_font()
style.apply_style(font)
LOG.info("Font: %s", font)

fig = draw_transferred_vs_human(subsets, rho)

style.save_panel(fig, LETTER, paths)
plt.close(fig)
LOG.info("Wrote %s", [str(path) for name, path in paths.items() if name != "provenance"])

# The saved PNG rather than the live figure, so what shows here is the file itself.
display(Image(filename=str(paths["png"]), width=WIDTH_SHOWN))

In [ ]:
record = style.provenance(
    notebook,
    parameters={"REPO": str(repo), "HUMAN_CELLS": HUMAN_CELLS, "OUT": OUT, "FORCE": FORCE},
    inputs={"human_cells": repo / HUMAN_CELLS},
    outputs=[paths["png"], paths["png_600dpi"], paths["svg"]],
    font=font,
)
record["panel"] = style.panel_spec(LETTER)
record["spearman_rho"] = {name: round(float(value), 4) for name, value in rho.items()}
record["n"] = {name: len(cells) for name, cells in subsets.items()}
style.write_json(paths["provenance"], record)
LOG.info("Wrote %s", paths["provenance"])